In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
pd.plotting.register_matplotlib_converters()
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
#reading the data
forest_df = pd.read_csv('../input/forest-cover-type-prediction/train.csv')

test_df = pd.read_csv('../input/forest-cover-type-prediction/test.csv')

# Removing the 'Id' column as it's not required for analysis or model training
forest_df = forest_df.drop(columns=["Id"])
test_df = test_df.drop(columns = ["Id"])

# Displaying the dimensions of the DataFrame
print("Dimensions of DataFrame: ", forest_df.shape)

# Displaying the column names
print("Column Names: ", forest_df.columns.tolist())

# Displaying the first few rows of the DataFrame to verify changes
forest_df.head()

In [ ]:
# Inspecting the structure of the DataFrame for null entries
forest_df.info(verbose=True, show_counts=True)

In [ ]:
#Cover Types 
cover_types = ['Spruce/Fir', 'Lodgepole Pine', 'Ponderosa Pine', 'Cottonwood/Willow', 'Aspen', 'Douglas Fir', 'Krummholz']

# Collecting counts for each wilderness area by cover type
counts_wilderness_area1 = np.array(forest_df.groupby('Cover_Type')['Wilderness_Area1'].sum())
counts_wilderness_area2 = np.array(forest_df.groupby('Cover_Type')['Wilderness_Area2'].sum())
counts_wilderness_area3 = np.array(forest_df.groupby('Cover_Type')['Wilderness_Area3'].sum())
counts_wilderness_area4 = np.array(forest_df.groupby('Cover_Type')['Wilderness_Area4'].sum())

#creating subplots
fig, ax = plt.subplots(ncols = 2, nrows = 2, figsize = (10,12))
plt.tight_layout()

#wilderness area 1
wedges, texts, autotexts = ax[0,0].pie(counts_wilderness_area1, labels=cover_types, autopct='%1.1f%%')

threshold = 1
for wedge, label, pct_label in zip(wedges, texts, autotexts):
    pct_value = pct_label.get_text().rstrip('%')
    if float(pct_value) < threshold:
        label.set_text('')       # remove text label
        pct_label.set_text('')   # remove percentage label
        wedge.set_visible(False) # remove wedge from pie
        wedge.set_label('')      # ensure wedge label does not go into legend

ax[0,0].legend(bbox_to_anchor=(0, 1))

ax[0,0].title.set_text('Wilderness Area 1')

#wilderness area 2
wedges, texts, autotexts = ax[0,1].pie(counts_wilderness_area2, labels=cover_types, autopct='%1.1f%%')

threshold = 1
for wedge, label, pct_label in zip(wedges, texts, autotexts):
    pct_value = pct_label.get_text().rstrip('%')
    if float(pct_value) < threshold:
        label.set_text('')       # remove text label
        pct_label.set_text('')   # remove percentage label
        wedge.set_visible(False) # remove wedge from pie
        wedge.set_label('')      # ensure wedge label does not go into legend

ax[0,1].legend(bbox_to_anchor=(1, 1))

ax[0,1].title.set_text('Wilderness Area 2')

#wilderness area 3
wedges, texts, autotexts = ax[1,0].pie(counts_wilderness_area3, labels=cover_types, autopct='%1.1f%%')

threshold = 1
for wedge, label, pct_label in zip(wedges, texts, autotexts):
    pct_value = pct_label.get_text().rstrip('%')
    if float(pct_value) < threshold:
        label.set_text('')       # remove text label
        pct_label.set_text('')   # remove percentage label
        wedge.set_visible(False) # remove wedge from pie
        wedge.set_label('')      # ensure wedge label does not go into legend

ax[1,0].legend(bbox_to_anchor=(0, 0))

ax[1,0].title.set_text('Wilderness Area 3')

#wilderness area 4
wedges, texts, autotexts = ax[1,1].pie(counts_wilderness_area4, labels=cover_types, autopct='%1.1f%%')

threshold = 1
for wedge, label, pct_label in zip(wedges, texts, autotexts):
    pct_value = pct_label.get_text().rstrip('%')
    if float(pct_value) < threshold:
        label.set_text('')       # remove text label
        pct_label.set_text('')   # remove percentage label
        wedge.set_visible(False) # remove wedge from pie
        wedge.set_label('')      # ensure wedge label does not go into legend

ax[1,1].legend(bbox_to_anchor=(1.2,0))

ax[1,1].title.set_text('Wilderness Area 4')

In [ ]:
#melting wilderness area booleans into a single categorical column 
wilderness_df = forest_df[['Wilderness_Area1', 'Wilderness_Area2', 'Wilderness_Area3',
       'Wilderness_Area4']]

wilderness_df = wilderness_df.dot(wilderness_df.columns).to_frame('Wilderness_Area')

#wilderness_df.head()

forest_merged = pd.concat([wilderness_df,forest_df['Cover_Type']], axis = 1, ignore_index=False)

forest_merged['Cover_Type'] = forest_merged.Cover_Type.astype(str)

#forest_merged.head()

#visualizing training data
sns.catplot(data=forest_merged, x="Wilderness_Area", y="Cover_Type")
plt.xticks(rotation=45)

In [ ]:
# Importing sklearn functionalities for model evaluation, preprocessing, and model selection
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, 
                             precision_score, recall_score, roc_auc_score, 
                             roc_curve, precision_recall_curve)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

In [ ]:
#Feature Engineering 

#total distance to nearest surface water feature (Pythagorean Theorem)
forest_df['Total_Distance_To_Hydrology'] = (forest_df['Vertical_Distance_To_Hydrology']**2 + forest_df['Horizontal_Distance_To_Hydrology']**2)**0.5
test_df['Total_Distance_To_Hydrology'] = (test_df['Vertical_Distance_To_Hydrology']**2 + test_df['Horizontal_Distance_To_Hydrology']**2)**0.5

forest_df.head(10)

In [ ]:
# Feature
X = forest_df.drop('Cover_Type', axis=1)

# Target
y = forest_df["Cover_Type"]
print(f"Shape of X: {X.shape}")
print(f"Shape of y: {y.shape}")

#split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_test: {y_test.shape}")

In [ ]:
#Scaling Features (normalizing ranges between features)
numerical_variables = ["Elevation", "Aspect", "Slope", "Horizontal_Distance_To_Hydrology", "Vertical_Distance_To_Hydrology", "Horizontal_Distance_To_Roadways", "Horizontal_Distance_To_Fire_Points", "Hillshade_9am", "Hillshade_Noon", "Hillshade_3pm", "Total_Distance_To_Hydrology"]

scaler = StandardScaler()
scaler.fit(X_train[numerical_variables])

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numerical_variables] = scaler.transform(X_train_scaled[numerical_variables])
X_test_scaled[numerical_variables] = scaler.transform(X_test_scaled[numerical_variables])

In [ ]:
#Model Evaluation

def run_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')

    print('Accuracy:', accuracy)
    print("F1 score (weighted average): ", f1)
    print("Precision (weighted average): ", precision)
    print("Recall (weighted average): ", recall)
    
    return y_pred

In [ ]:
#Fitting Model 
forest_model = RandomForestClassifier(random_state=1)
forest_model.fit(X_train_scaled, y_train)

#Making Predictions
y_pred = run_model(forest_model, X_test, y_test)